# 3. Gated Recurrent Architecture: Bidirectional LSTM
**CSE 4122 — Natural Language Processing Laboratory**  
*Department of Computer Science and Engineering, Khulna University of Engineering & Technology (KUET)*

---

### Overview
This notebook implements a Bidirectional Long Short-Term Memory (Bi-LSTM) network for sarcasm detection:
- **Gating Mechanism**: Solves vanishing/exploding gradients through input, forget, and output gates.
- **Bidirectionality**: Processes text in both forward (left-to-right) and backward (right-to-left) directions to capture context shifts typical of sarcasm.
- **Regularization**: Dropout layers and weight decay.
- **Evaluation**: Classification report, confusion matrix, and threshold optimization.


## 1. Setup & Imports

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix, roc_auc_score

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[OK] Device configured: {device}")


## 2. Data Loading & Sequence Pipeline

In [ ]:
train_path = os.path.join("dataset", "train.csv")
test_path = os.path.join("dataset", "test_1.csv")

if not os.path.exists(train_path):
    train_path = "train.csv"
if not os.path.exists(test_path):
    test_path = "test_1.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

t_col_train = "tweet" if "tweet" in train_df.columns else "text"
t_col_test = "tweet" if "tweet" in test_df.columns else "text"

def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'@[A-Za-z0-9_]+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train_texts = [clean_text(t) for t in train_df[t_col_train]]
train_labels = train_df["sarcastic"].astype(int).tolist()

test_texts = [clean_text(t) for t in test_df[t_col_test]]
test_labels = test_df["sarcastic"].astype(int).tolist()

# Vocabulary builder
class Tokenizer:
    def __init__(self, max_vocab: int = 6000, max_len: int = 64):
        self.max_vocab = max_vocab
        self.max_len = max_len
        self.vocab = {"<PAD>": 0, "<UNK>": 1}

    def fit(self, texts):
        c = Counter()
        for t in texts:
            c.update(re.findall(r'\b\w+\b', t.lower()))
        for idx, (w, _) in enumerate(c.most_common(self.max_vocab - 2), start=2):
            self.vocab[w] = idx

    def encode(self, text):
        words = re.findall(r'\b\w+\b', text.lower())
        ids = [self.vocab.get(w, 1) for w in words]
        if len(ids) < self.max_len:
            ids = ids + [0] * (self.max_len - len(ids))
        else:
            ids = ids[:self.max_len]
        return ids

tokenizer = Tokenizer(max_vocab=6000, max_len=64)
tokenizer.fit(train_texts)

class LSTMDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return torch.tensor(self.tokenizer.encode(self.texts[idx]), dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)

train_loader = DataLoader(LSTMDataset(train_texts, train_labels, tokenizer), batch_size=32, shuffle=True)
test_loader = DataLoader(LSTMDataset(test_texts, test_labels, tokenizer), batch_size=64, shuffle=False)
print(f"Vocab size: {len(tokenizer.vocab)} | Train batches: {len(train_loader)} | Test batches: {len(test_loader)}")


## 3. Bidirectional LSTM Architecture
Features:
- `nn.Embedding`: 128-dimensional learned word representations.
- `nn.LSTM`: Bidirectional LSTM with hidden size 128 (yielding 256-dimensional concatenated forward + backward states).
- Multi-layer classification head with `nn.Dropout` to prevent overfitting.


In [ ]:
class SarcasmBiLSTM(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int = 128, hidden_dim: int = 128, num_classes: int = 2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.4)
        self.fc1 = nn.Linear(hidden_dim * 2, 64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, num_classes)

    def forward(self, x):
        embeds = self.embedding(x)
        output, (hidden, cell) = self.lstm(embeds)
        # Concatenate final forward and backward states
        forward_final = hidden[-2]
        backward_final = hidden[-1]
        concat_hidden = torch.cat((forward_final, backward_final), dim=1)
        
        x = self.dropout(concat_hidden)
        x = self.relu(self.fc1(x))
        logits = self.fc2(x)
        return logits

bilstm_model = SarcasmBiLSTM(vocab_size=len(tokenizer.vocab)).to(device)
print(bilstm_model)


## 4. Model Training with Class Weighting

In [ ]:
# Compute class weights
n_0 = train_labels.count(0)
n_1 = train_labels.count(1)
weights = torch.tensor([len(train_labels)/(2.0*n_0), len(train_labels)/(2.0*n_1)], dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.AdamW(bilstm_model.parameters(), lr=1e-3, weight_decay=1e-3)

num_epochs = 7
loss_history = []

print("Training Bi-LSTM...")
bilstm_model.train()
for epoch in range(num_epochs):
    epoch_loss = 0.0
    for x_b, y_b in train_loader:
        x_b, y_b = x_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        out = bilstm_model(x_b)
        loss = criterion(out, y_b)
        loss.backward()
        nn.utils.clip_grad_norm_(bilstm_model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader)
    loss_history.append(avg_loss)
    print(f"Epoch {epoch+1}/{num_epochs} — Loss: {avg_loss:.4f}")

print("[OK] Training finished.")


## 5. Test Evaluation & Confusion Matrix

In [ ]:
bilstm_model.eval()
test_preds, test_probs, test_targets = [], [], []

with torch.no_grad():
    for x_b, y_b in test_loader:
        x_b = x_b.to(device)
        logits = bilstm_model(x_b)
        p = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        test_probs.extend(p)
        test_preds.extend((p >= 0.5).astype(int))
        test_targets.extend(y_b.numpy())

test_targets = np.array(test_targets)
test_preds = np.array(test_preds)
test_probs = np.array(test_probs)

acc = accuracy_score(test_targets, test_preds)
prec, rec, f1, _ = precision_recall_fscore_support(test_targets, test_preds, average="binary", zero_division=0)
roc_auc = roc_auc_score(test_targets, test_probs)

print("="*45)
print("           Bi-LSTM Test Metrics")
print("="*45)
print(f"Accuracy:   {acc:.4f}")
print(f"Precision:  {prec:.4f}")
print(f"Recall:     {rec:.4f}")
print(f"F1 Score:   {f1:.4f}")
print(f"ROC-AUC:    {roc_auc:.4f}")
print("="*45)
print("\nClassification Report:\n", classification_report(test_targets, test_preds, target_names=["Non-Sarcastic", "Sarcastic"]))

cm = confusion_matrix(test_targets, test_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens",
            xticklabels=["Non-Sarcastic", "Sarcastic"],
            yticklabels=["Non-Sarcastic", "Sarcastic"])
plt.title("Bi-LSTM Test Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.show()


## 6. Threshold Optimization

In [ ]:
for th in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    p = (test_probs >= th).astype(int)
    a = accuracy_score(test_targets, p)
    pr, rc, f, _ = precision_recall_fscore_support(test_targets, p, average="binary", zero_division=0)
    print(f"Threshold {th:.2f} -> Accuracy: {a:.4f}, Precision: {pr:.4f}, Recall: {rc:.4f}, F1: {f:.4f}")
